# Huấn luyện, Hiệu chỉnh Ngưỡng và Đánh giá CAE trên Google Colab

**Đề tài:** Xây dựng và đánh giá mạng Convolutional Autoencoder trong phát hiện và khoanh vùng bất thường bề mặt sản phẩm công nghiệp.

> **Quy trình hoạt động:**
> - Logic thuật toán được quản lý trong các module `.py` trên GitHub.
> - Notebook này vừa đóng vai trò **bảng điều khiển (launcher)** gọi các lệnh thực thi GPU, vừa là **bảng hiển thị trực quan (dashboard)** để bạn xem đồ thị loss, bảng điểm số và các bức ảnh phát hiện/khoanh vùng lỗi ngay trên trình duyệt.

---
## 1. Kết nối Google Drive & Kiểm tra GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
!python -c "import torch; print('PyTorch:', torch.__version__, '| CUDA Available:', torch.cuda.is_available(), '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"

---
## 2. Đồng bộ mã nguồn từ GitHub & Cài đặt thư viện

*(Nếu là lần đầu: clone repo. Các lần sau: chỉ cần `git pull` để nhận code mới nhất từ máy cá nhân)*

In [ ]:
import os

REPO_URL = "https://github.com/Trong59/industrial-anomaly-cae.git"
REPO_DIR = "/content/industrial-anomaly-cae"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

!pip install -r requirements.txt -q
print("[*] Đã đồng bộ mã nguồn và cài đặt thư viện thành công!")

---
## 3. Chuẩn bị Dữ liệu & Xem trước Ảnh Mẫu (Data Preview)

Sao chép danh mục `bottle` từ Google Drive sang ổ đĩa tạm `/content/` để nạp dữ liệu nhanh nhất.

In [ ]:
# Tạo thư mục tạm trên Colab runtime
!mkdir -p /content/cae_data/mvtec_anomaly_detection

DRIVE_DATASET_PATH = "/content/drive/MyDrive/CAE_Project/datasets/mvtec_anomaly_detection/bottle"
LOCAL_DATASET_PATH = "/content/cae_data/mvtec_anomaly_detection/bottle"

# Sao chép danh mục sang /content/
!cp -r "{DRIVE_DATASET_PATH}" "/content/cae_data/mvtec_anomaly_detection/"
print("[*] Dữ liệu đã sẵn sàng tại:", LOCAL_DATASET_PATH)

In [ ]:
# Trực quan hóa: Xem trước ảnh bình thường và ảnh lỗi trong tập dữ liệu
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

base_path = Path(LOCAL_DATASET_PATH)
train_good = sorted(list((base_path / "train" / "good").glob("*.png")))
test_defect = sorted(list((base_path / "test" / "broken_large").glob("*.png")))
gt_defect = sorted(list((base_path / "ground_truth" / "broken_large").glob("*.png")))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
if train_good:
    axes[0].imshow(Image.open(train_good[0]))
    axes[0].set_title(f"Train Normal (Good)\n{train_good[0].name}")
axes[0].axis("off")

if test_defect:
    axes[1].imshow(Image.open(test_defect[0]))
    axes[1].set_title(f"Test Anomaly (Defect)\n{test_defect[0].name}")
axes[1].axis("off")

if gt_defect:
    axes[2].imshow(Image.open(gt_defect[0]), cmap="gray")
    axes[2].set_title(f"Ground Truth Mask\n{gt_defect[0].name}")
axes[2].axis("off")

plt.suptitle("Kiểm tra tính toàn vẹn của Dữ liệu MVTec AD Bottle", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4. Kiểm tra Kiến trúc Mạng Convolutional Autoencoder (CAE)

In [ ]:
import torch
from src.models.cae import ConvolutionalAutoencoder

model = ConvolutionalAutoencoder(in_channels=3, base_channels=32, output_activation="sigmoid")
dummy_input = torch.randn(1, 3, 256, 256)
dummy_output = model(dummy_input)

print("=== KIẾN TRÚC MẠNG CONVOLUTIONAL AUTOENCODER ===")
print(f"- Input Shape:         {list(dummy_input.shape)}")
print(f"- Reconstructed Shape: {list(dummy_output.shape)}")
print(f"- Tổng số tham số:    {model.count_parameters():,} parameters")

---
## 5. Huấn luyện Baseline E1 (MSE Loss, No Bottleneck)

Huấn luyện trên ảnh bình thường `train/good`. Checkpoint và log được lưu vào Google Drive.

In [ ]:
!python train.py \
  --config configs/bottle.yaml \
  --data-root /content/cae_data/mvtec_anomaly_detection \
  --output-root /content/drive/MyDrive/CAE_Project/outputs \
  --experiment-id E1_bottle_mse_seed42

In [ ]:
# Hiển thị Biểu đồ Loss Curve của E1
from IPython.display import Image, display
import pandas as pd

exp_dir = "/content/drive/MyDrive/CAE_Project/outputs/E1_bottle_mse_seed42"
loss_img_path = f"{exp_dir}/plots/bottle_loss_curve.png"
log_csv_path = f"{exp_dir}/logs/bottle_train.csv"

if os.path.exists(loss_img_path):
    display(Image(filename=loss_img_path))
if os.path.exists(log_csv_path):
    df_log = pd.read_csv(log_csv_path)
    print(f"[*] Best Val Loss E1: {df_log['val_loss'].min():.6f} (Epoch {df_log.loc[df_log['val_loss'].idxmin(), 'epoch']})")

---
## 6. Hiệu chỉnh Ngưỡng cho E1 (calibrate.py)

In [ ]:
!python calibrate.py \
  --config configs/bottle.yaml \
  --data-root /content/cae_data/mvtec_anomaly_detection \
  --output-root /content/drive/MyDrive/CAE_Project/outputs \
  --experiment-id E1_bottle_mse_seed42

---
## 7. Đánh giá Test cho Baseline E1 (evaluate.py)

In [ ]:
!python evaluate.py \
  --config configs/bottle.yaml \
  --data-root /content/cae_data/mvtec_anomaly_detection \
  --output-root /content/drive/MyDrive/CAE_Project/outputs \
  --experiment-id E1_bottle_mse_seed42 \
  --max-vis-samples 4

---
## 8. THÍ NGHIỆM MỞ RỘNG E2 (L1 Loss + Bottleneck Nén 64 + Ngưỡng 95%)

**Cải tiến khoa học:**
- **L1 Loss:** Giúp bảo toàn gradient sắc nét ở biên các khuyết tật nhỏ.
- **Bottleneck Nén 64:** Giảm số chiều biểu diễn đặc trưng ở đáy mạng để triệt tiêu hiện tượng CAE tái tạo luôn cả vết nứt.
- **Ngưỡng Percentile 95%:** Tăng độ nhạy phát hiện bất thường, giảm thiểu 38 ca False Negative ở E1.

In [ ]:
# 8.1. Huấn luyện mô hình E2
!python train.py \
  --config configs/bottle_e2.yaml \
  --data-root /content/cae_data/mvtec_anomaly_detection \
  --output-root /content/drive/MyDrive/CAE_Project/outputs \
  --experiment-id E2_bottle_l1_bottleneck_seed42

In [ ]:
# 8.2. Hiệu chỉnh ngưỡng cho E2 trên tập validation
!python calibrate.py \
  --config configs/bottle_e2.yaml \
  --data-root /content/cae_data/mvtec_anomaly_detection \
  --output-root /content/drive/MyDrive/CAE_Project/outputs \
  --experiment-id E2_bottle_l1_bottleneck_seed42

In [ ]:
# 8.3. Đánh giá độc lập E2 trên tập test
!python evaluate.py \
  --config configs/bottle_e2.yaml \
  --data-root /content/cae_data/mvtec_anomaly_detection \
  --output-root /content/drive/MyDrive/CAE_Project/outputs \
  --experiment-id E2_bottle_l1_bottleneck_seed42 \
  --max-vis-samples 4

---
## 9. BẢNG SO SÁNH ĐỐI CHỨNG THỰC NGHIỆM: E1 (Baseline) vs E2 (Cải tiến)

In [ ]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

e1_path = "/content/drive/MyDrive/CAE_Project/outputs/E1_bottle_mse_seed42/metrics/bottle_metrics.json"
e2_path = "/content/drive/MyDrive/CAE_Project/outputs/E2_bottle_l1_bottleneck_seed42/metrics/bottle_metrics.json"

if os.path.exists(e1_path) and os.path.exists(e2_path):
    with open(e1_path) as f1, open(e2_path) as f2:
        m1 = json.load(f1)
        m2 = json.load(f2)

    comp_dict = {
        "Chỉ số (Metric)": [
            "Image-level AUROC",
            "Image-level F1-Score",
            "Precision",
            "Recall (Độ nhạy bắt lỗi)",
            "True Positives (Số lỗi bắt đúng)",
            "False Negatives (Số lỗi bị sót)",
            "False Positives (Báo sai ảnh tốt)",
            "Pixel-level AUROC",
            "Pixel-level IoU",
            "Pixel-level Dice"
        ],
        "E1: Baseline (MSE, No Bottleneck)": [
            f"{m1['image_level']['auroc']:.4f}",
            f"{m1['image_level']['f1']:.4f}",
            f"{m1['image_level']['precision']:.4f}",
            f"{m1['image_level']['recall']:.4f}",
            m1['image_level']['confusion_matrix']['tp'],
            m1['image_level']['confusion_matrix']['fn'],
            m1['image_level']['confusion_matrix']['fp'],
            f"{m1['pixel_level']['pixel_auroc']:.4f}",
            f"{m1['pixel_level']['pixel_iou']:.4f}",
            f"{m1['pixel_level']['pixel_dice']:.4f}",
        ],
        "E2: Cải tiến (L1 + Bottleneck 64)": [
            f"{m2['image_level']['auroc']:.4f}",
            f"{m2['image_level']['f1']:.4f}",
            f"{m2['image_level']['precision']:.4f}",
            f"{m2['image_level']['recall']:.4f}",
            m2['image_level']['confusion_matrix']['tp'],
            m2['image_level']['confusion_matrix']['fn'],
            m2['image_level']['confusion_matrix']['fp'],
            f"{m2['pixel_level']['pixel_auroc']:.4f}",
            f"{m2['pixel_level']['pixel_iou']:.4f}",
            f"{m2['pixel_level']['pixel_dice']:.4f}",
        ]
    }
    df_comp = pd.DataFrame(comp_dict)
    print("===========================================================")
    print("           BẢNG SO SÁNH ĐỐI CHỨNG THỰC NGHIỆM E1 vs E2      ")
    print("===========================================================")
    display(df_comp)

    e1_roc = "/content/drive/MyDrive/CAE_Project/outputs/E1_bottle_mse_seed42/plots/bottle_image_roc.png"
    e2_roc = "/content/drive/MyDrive/CAE_Project/outputs/E2_bottle_l1_bottleneck_seed42/plots/bottle_image_roc.png"
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    if os.path.exists(e1_roc):
        axes[0].imshow(plt.imread(e1_roc))
        axes[0].set_title("E1: Baseline MSE ROC")
    axes[0].axis("off")
    if os.path.exists(e2_roc):
        axes[1].imshow(plt.imread(e2_roc))
        axes[1].set_title("E2: L1 + Bottleneck 64 ROC")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Vui lòng chạy xong cả E1 và E2 để xem bảng so sánh!")